In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import jax
from ase.visualize import view
from ase.atoms import Atoms

from msmjax.core.shortrange import make_eval_pair_pot, _gen_supercell
from msmjax.utils.benchmarking import (
    eval_lammps_pppm,
    path_input_structures,
)

LAMMPS_EXECUTABLE = "/home/florian/Downloads/lammps-static/bin/lmp"

In [2]:
# TODO: Also compute reference results for stress? (LAMMPS compute pressure command?)
# TODO: Charge gradient (= electrostatic potential at particle positions)?

# Function definitions

In [3]:
def coulomb_kernel(r):
    return 1.0 / r


def calc_energy_ref_nonperiodic(positions, charges):
    n_dim = positions.shape[1]
    compute_pair_term = make_eval_pair_pot(
        kernel_fn=coulomb_kernel, pbc=(False,) * n_dim
    )
    return compute_pair_term(positions, charges)


# TODO
# @jax.jit
# def calc_reference_results_nonperiodic(positions, charges):
#     value, grad = jax.value_and_grad(
#         calc_energy_ref_nonperiodic, argnums=(0, 1)
#     )(positions, charges)
#     forces = -grad[0]
#     charge_gradient = grad[1]
#     return value, forces, charge_gradient


def calc_reference_forces_nonperiodic(positions, charges):
    return -jax.grad(calc_energy_ref_nonperiodic, argnums=0)(
        positions, charges
    )


def calc_reference_chargegrad_nonperiodic(positions, charges):
    return jax.grad(calc_energy_ref_nonperiodic, argnums=1)(positions, charges)


def calc_reference_results_nonperiodic(positions, charges):
    # This is less likely to run out of memory than calculating everything
    # with a single jax.value_and_grad call
    value = jax.jit(calc_energy_ref_nonperiodic)(positions, charges)
    forces = jax.jit(calc_reference_forces_nonperiodic)(positions, charges)
    chargegrad = jax.jit(calc_reference_chargegrad_nonperiodic)(
        positions, charges
    )
    return value, forces, chargegrad

In [4]:
def load_one_structure(n_particles):
    structures = onp.load(
        path_input_structures / ("structures_" + str(n_particles) + ".npz")
    )
    # TODO: Also test different structures of the same number of particles?
    #  (i.e., other values for idx_structure than 0)
    idx_structure = 0
    pos = structures["positions"][idx_structure]
    chg = structures["charges"][idx_structure]
    cell = structures["cells"][idx_structure]
    pos = pos.astype(onp.float64)
    chg = chg.astype(onp.float64)
    cell = cell.astype(onp.float64)
    return pos, chg, cell

# Non-periodic

## Cubic

In [5]:
(pos, chg, cell) = load_one_structure(10000)

e_ref, f_ref, chargegrad_ref = calc_reference_results_nonperiodic(pos, chg)

fname = "nonperiodic_cubic.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
)

## Orthorhombic with different side lengths

The intention is that the side lengths are so different that the minimum number of grid points is reached along some axis before the others.

In [6]:
(pos, chg, cell) = load_one_structure(1500)
(pos, chg, cell) = _gen_supercell(pos, chg, cell, supercell_diag=(3, 2, 1))
scaled_pos = pos @ onp.linalg.pinv(cell)
axis_stretch_factors = onp.array([1.1, 1.0, 0.9])
cell *= axis_stretch_factors
pos = scaled_pos @ cell

e_ref, f_ref, chargegrad_ref = calc_reference_results_nonperiodic(pos, chg)

fname = "nonperiodic_ortho-different-sidelengths.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
)

atoms = Atoms(positions=pos, cell=cell, charges=chg)
view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

## Triclinic

In [7]:
(pos, chg, cell) = load_one_structure(10000)

atoms = Atoms(positions=pos, charges=chg, cell=cell)
new_lengths = onp.diag(cell) * (0.8, 1.0, 1.25)
new_angles = [75, 90, 120]
nonortho_cell = onp.concatenate([new_lengths, new_angles])
atoms.set_cell(nonortho_cell, scale_atoms=True)
pos = onp.array(atoms.get_positions())
chg = onp.array(atoms.get_initial_charges())
cell = onp.array(atoms.get_cell()[...])

e_ref, f_ref, chargegrad_ref = calc_reference_results_nonperiodic(pos, chg)

fname = "nonperiodic_triclinic.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
)

view(atoms)

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

# Periodic

## Cubic

In [8]:
(pos, chg, cell) = load_one_structure(500)

e_ref, f_ref, chargegrad_ref, stress_ref = eval_lammps_pppm(
    pos, chg, cell, LAMMPS_EXECUTABLE, max_neighbors_one_atom=10000
)

fname = "periodic_cubic.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
    stress=stress_ref,
)

## Orthorhombic with different side lengths

The intention is that the side lengths are so different that the grid is reduced to a single point along some axis faster than along the others.

In [9]:
(pos, chg, cell) = load_one_structure(500)
(pos, chg, cell) = _gen_supercell(pos, chg, cell, supercell_diag=(3, 2, 1))
scaled_pos = pos @ onp.linalg.pinv(cell)
axis_stretch_factors = onp.array([1.1, 1.0, 0.9])
cell *= axis_stretch_factors
pos = scaled_pos @ cell

e_ref, f_ref, chargegrad_ref, stress_ref = eval_lammps_pppm(
    pos,
    chg,
    cell,
    LAMMPS_EXECUTABLE,
    max_neighbors_one_atom=10000,
    show_stdout=True,
)

fname = "periodic_ortho-different-sidelengths.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
    stress=stress_ref,
)

atoms = Atoms(positions=pos, cell=cell, charges=chg)
view(atoms)

LAMMPS (27 Jun 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
Reading data file ...
  orthogonal box = (0 0 0) to (26.192117 15.87401 7.1433045)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  3000 atoms
  read_data CPU = 0.013 seconds

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

Your simulation uses code contributions which should be cited:
- Type Label Framework: https://doi.org/10.1021/acs.jpcb.3c08419
The log file lists these citations in BibTeX format.

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

PPPM initialization ...
  using 12-bit tables for long-range coulomb (src/kspace.cpp:342)
  G vector (1/distance) = 0.32175092
  grid = 24 18 10
  stencil order = 5
  estimated absolute RMS force accuracy = 0.00013123815
  estimated relative force accuracy = 9.1139854e-06
  using double precision KISS FFT
  3d grid and FFT values/proc = 13175 4320
Generated 0

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

## Triclinic

In [10]:
(pos, chg, cell) = load_one_structure(500)

atoms = Atoms(positions=pos, charges=chg, cell=cell)
new_lengths = onp.diag(cell) * (0.9, 1.0, 1.25)
new_angles = [75, 90, 120]
nonortho_cell = onp.concatenate([new_lengths, new_angles])
atoms.set_cell(nonortho_cell, scale_atoms=True)
pos = onp.array(atoms.get_positions())
chg = onp.array(atoms.get_initial_charges())
cell = onp.array(atoms.get_cell()[...])

e_ref, f_ref, chargegrad_ref, stress_ref = eval_lammps_pppm(
    pos,
    chg,
    cell,
    LAMMPS_EXECUTABLE,
    max_neighbors_one_atom=10000,
    show_stdout=True,
)

fname = "periodic_triclinic.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
    stress=stress_ref,
)

view(atoms)

LAMMPS (27 Jun 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
Reading data file ...
  triclinic box = (0 0 0) to (7.1433045 6.873648 9.4678295) with tilt (-3.9685025 0 2.9650517)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  500 atoms
  read_data CPU = 0.002 seconds

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

Your simulation uses code contributions which should be cited:
- Type Label Framework: https://doi.org/10.1021/acs.jpcb.3c08419
The log file lists these citations in BibTeX format.

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

PPPM initialization ...
  using 12-bit tables for long-range coulomb (src/kspace.cpp:342)
  G vector (1/distance) = 0.29278275
  grid = 10 15 15
  stencil order = 5
  estimated absolute RMS force accuracy = 0.00084001054
  estimated relative force accuracy = 5.8335504e-05
  using double precision KISS FFT
  3d grid and FFT val

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>